In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
df_cleaned_resume = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_resume.pkl')
df_cleaned_jd = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_jd.pkl')
similarity_matrix = np.load('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/similarity_matrix.npy')

In [3]:
similarity_matrix

array([[0.04810346, 0.06250257, 0.055381  , 0.10244972, 0.08105191,
        0.04565973],
       [0.0436544 , 0.05078145, 0.10209577, 0.06435448, 0.09000305,
        0.04596838],
       [0.08228704, 0.05073942, 0.06196083, 0.095364  , 0.14315012,
        0.06093925],
       ...,
       [0.02449801, 0.01602976, 0.09582703, 0.02033288, 0.03376689,
        0.05546926],
       [0.01582061, 0.00819284, 0.05138346, 0.02077143, 0.01470102,
        0.01057604],
       [0.04851061, 0.02296338, 0.21028638, 0.04269504, 0.04834374,
        0.04763549]])

In [4]:
print(df_cleaned_resume.columns.tolist())
print()
print(df_cleaned_jd.columns.tolist())

['ID', 'Resume_str', 'Resume_html', 'Category', 'cleaned_resume', 'extracted_skills', 'best_match_category', 'best_match_score']

['category', 'job_title', 'job_description', 'cleaned_description', 'extracted_skills']


In [5]:
def skill_overlap_score(resume_skills, jd_skills):
  if (len(jd_skills) == 0):
    return 0
  overlap = set(resume_skills) & set(jd_skills)
  overlap_score = len(overlap)/len(jd_skills)
  return overlap_score

In [6]:
#given ONE job category, it ranks all resumes against it
def rank_resume_for_job(jd_category, top_n=10):
    jd_idx = list(df_cleaned_jd['category']).index(jd_category)
    jd_skills = df_cleaned_jd.iloc[jd_idx]['extracted_skills']

    results = df_cleaned_resume.copy()
    results['similarity_score'] = similarity_matrix[:, jd_idx]
    results['skill_overlap'] = results['extracted_skills'].apply(lambda skills: skill_overlap_score(skills, jd_skills))

    results['combined_score'] = (0.7 * results['similarity_score']) + (0.3 * results['skill_overlap'])

    #using min-max scaling to scale the results
    min_score = results['combined_score'].min()
    max_score = results['combined_score'].max()

    results['match_percentage'] = (((results['combined_score'] - min_score)/(max_score - min_score))*100).round(1)

    ranked = results.sort_values('combined_score', ascending=False)
    return ranked[['Category', 'similarity_score', 'skill_overlap', 'combined_score', 'match_percentage']].head(top_n)

In [7]:
rank_resume_for_job('HR', 7)

,Category,similarity_score,skill_overlap,combined_score,match_percentage
11,HR,0.205748,1.000000,0.444024,100.0
23,HR,0.203236,1.000000,0.442265,99.6
81,HR,0.256345,0.857143,0.436584,98.3
67,HR,0.188630,1.000000,0.432041,97.3
85,HR,0.187834,1.000000,0.431484,97.2
19,HR,0.186339,1.000000,0.430438,96.9
4,HR,0.184868,1.000000,0.429408,96.7


In [8]:
rank_resume_for_job('ENGINEERING', 7)

,Category,similarity_score,skill_overlap,combined_score,match_percentage
1776,ENGINEERING,0.178953,1.000000,0.425267,100.0
1787,ENGINEERING,0.164444,1.000000,0.415111,97.6
1084,SALES,0.147829,1.000000,0.403480,94.8
1803,ENGINEERING,0.201775,0.833333,0.391242,91.9
1737,ENGINEERING,0.125890,1.000000,0.388123,91.1
1720,ENGINEERING,0.189211,0.833333,0.382448,89.8
1709,ENGINEERING,0.177210,0.833333,0.374047,87.8


In [9]:
rank_resume_for_job('INFORMATION-TECHNOLOGY', 7)

,Category,similarity_score,skill_overlap,combined_score,match_percentage
281,INFORMATION-TECHNOLOGY,0.119716,1.0,0.383801,100.0
234,INFORMATION-TECHNOLOGY,0.078401,1.0,0.354881,92.3
1091,SALES,0.078060,1.0,0.354642,92.3
225,INFORMATION-TECHNOLOGY,0.073692,1.0,0.351584,91.5
283,INFORMATION-TECHNOLOGY,0.073670,1.0,0.351569,91.5
223,INFORMATION-TECHNOLOGY,0.072345,1.0,0.350642,91.2
258,INFORMATION-TECHNOLOGY,0.071356,1.0,0.349949,91.0


In [10]:
rank_resume_for_job('HEALTHCARE', 7)

,Category,similarity_score,skill_overlap,combined_score,match_percentage
761,HEALTHCARE,0.308779,0.857143,0.473288,100.0
791,HEALTHCARE,0.351922,0.714286,0.460631,97.3
745,HEALTHCARE,0.341082,0.714286,0.453043,95.7
783,HEALTHCARE,0.318176,0.714286,0.437009,92.3
693,HEALTHCARE,0.216364,0.857143,0.408598,86.2
786,HEALTHCARE,0.192262,0.857143,0.391727,82.7
731,HEALTHCARE,0.191291,0.857143,0.391046,82.5


In [11]:
rank_resume_for_job('SALES', 7)

,Category,similarity_score,skill_overlap,combined_score,match_percentage
1085,SALES,0.106604,1.0,0.374623,100.0
1014,SALES,0.103024,1.0,0.372117,99.3
1088,SALES,0.098618,1.0,0.369032,98.5
1081,SALES,0.095141,1.0,0.366599,97.8
1078,SALES,0.089290,1.0,0.362503,96.7
1080,SALES,0.086644,1.0,0.360651,96.2
1016,SALES,0.084690,1.0,0.359283,95.8


In [12]:
rank_resume_for_job('FINANCE', 7)

,Category,similarity_score,skill_overlap,combined_score,match_percentage
1493,FINANCE,0.313636,0.909091,0.492273,100.0
1524,FINANCE,0.279308,0.909091,0.468243,95.1
1557,FINANCE,0.277734,0.909091,0.467141,94.9
1505,FINANCE,0.235703,1.000000,0.464992,94.4
1483,FINANCE,0.272977,0.909091,0.463811,94.2
1585,FINANCE,0.255190,0.909091,0.451360,91.7
1491,FINANCE,0.288838,0.818182,0.447641,90.9


In [13]:
IT_jd_skills = df_cleaned_jd[df_cleaned_jd['category']=='INFORMATION-TECHNOLOGY']['extracted_skills'].iloc[0]
print(IT_jd_skills)
print(len(IT_jd_skills))

['microsoft', 'technical', 'desktop', 'application']
4


In [14]:
category_order = list(df_cleaned_jd['category'])
for cat in category_order:
  skills = df_cleaned_jd[df_cleaned_jd['category'] == cat]['extracted_skills'].iloc[0]
  print (cat, ' :', skills, '| count:', len(skills))

INFORMATION-TECHNOLOGY  : ['microsoft', 'technical', 'desktop', 'application'] | count: 4
FINANCE  : ['finance', 'accountant', 'balance', 'cash', 'reconciliation', 'variance', 'forecast', 'revenue', 'audit', 'accounting', 'financial statement'] | count: 11
ENGINEERING  : ['material', 'inspection', 'production', 'drawing', 'mechanical', 'manufacturing'] | count: 6
SALES  : ['customer service', 'stock', 'sell'] | count: 3
HR  : ['generalist', 'employee relation', 'human resource', 'compensation', 'interview', 'benefit', 'recruit'] | count: 7
HEALTHCARE  : ['insurance', 'treatment', 'patient', 'provider', 'hospital', 'nursing', 'nurse'] | count: 7


In [15]:
df_cleaned_resume['best_match_category_combined'] = None
for cat in category_order:
  jd_idx = list(df_cleaned_jd['category']).index(cat)
  jd_skills = df_cleaned_jd.iloc[jd_idx]['extracted_skills']


In [16]:
#it's checking "if I look at ALL 6 JDs simultaneously for each resume, does the highest-scoring one match the resume's true label
combined_matrix = np.zeros((len(df_cleaned_resume), len(category_order)))
for i, cat in enumerate(category_order):
    jd_idx = list(df_cleaned_jd['category']).index(cat)
    jd_skills = df_cleaned_jd.iloc[jd_idx]['extracted_skills']
    skill_overlaps = df_cleaned_resume['extracted_skills'].apply(lambda skills: skill_overlap_score(skills, jd_skills))
    combined_matrix[:, i] = (0.90 * similarity_matrix[:, i]) + (0.10 * skill_overlaps)

df_cleaned_resume['best_match_combined'] = [category_order[i] for i in combined_matrix.argmax(axis=1)]
combined_accuracy = (df_cleaned_resume['Category'] == df_cleaned_resume['best_match_combined']).mean()
print(f"Combined-score accuracy: {combined_accuracy:.2%}")

Combined-score accuracy: 75.18%


In [17]:
print(rank_resume_for_job('INFORMATION-TECHNOLOGY', 7))
print()
print(rank_resume_for_job('SALES', 7))

                    Category  similarity_score  skill_overlap  combined_score  \
281   INFORMATION-TECHNOLOGY          0.119716            1.0        0.383801   
234   INFORMATION-TECHNOLOGY          0.078401            1.0        0.354881   
1091                   SALES          0.078060            1.0        0.354642   
225   INFORMATION-TECHNOLOGY          0.073692            1.0        0.351584   
283   INFORMATION-TECHNOLOGY          0.073670            1.0        0.351569   
223   INFORMATION-TECHNOLOGY          0.072345            1.0        0.350642   
258   INFORMATION-TECHNOLOGY          0.071356            1.0        0.349949   

      match_percentage  
281              100.0  
234               92.3  
1091              92.3  
225               91.5  
283               91.5  
223               91.2  
258               91.0  

     Category  similarity_score  skill_overlap  combined_score  \
1085    SALES          0.106604            1.0        0.374623   
1014    SALES     

In [18]:
df_cleaned_resume.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_resume.pkl')
df_cleaned_jd.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_jd.pkl')
print("all saved after 3 days of debugging")

all saved after 3 days of debugging
